################ MD01: NOTEBOOK OVERVIEW ################
# WILLMA Whisper Transcriber V03

This notebook provides a local-PC batch transcription workflow for audio files stored in a folder.

It is organized as separate sections for:

- setup and notebook usage
- dependency handling
- configuration
- audio discovery and validation
- Whisper model loading
- single-file and batch transcription
- transcript formatting and saving
- progress and error reporting
- end-to-end execution
- preview and verification

The workflow is designed for use in Visual Studio Code with Jupyter support.

In [ ]:
################ CELL01: SETUP — PATHS, IMPORTS AND CONFIGURATION ################
from pathlib import Path

# ── Paths & output folders ────────────────────────────────────────────────────────
NOTEBOOK_PATH        = Path(r"D:\OneDrive - Hogeschool Rotterdam\SURF_PILOT\AI_HUB_PILOT\PYTHON\WILLMA_WHISPER_TRANSCRIBER_V03.ipynb")
DEFAULT_INPUT_FOLDER = Path(r"D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHISPER\WAVS")
DEFAULT_OUTPUT_FOLDER    = DEFAULT_INPUT_FOLDER / "transcriber_v03_outputs"
DEFAULT_CLEANED_FOLDER   = DEFAULT_OUTPUT_FOLDER / "cleaned_audio"
DEFAULT_LOG_FOLDER       = DEFAULT_OUTPUT_FOLDER / "logs"
DEFAULT_PROGRESS_FILE    = DEFAULT_LOG_FOLDER    / "batch_progress.json"
DEFAULT_REFERENCE_FOLDER = DEFAULT_OUTPUT_FOLDER / "references"

for _d in [DEFAULT_OUTPUT_FOLDER, DEFAULT_CLEANED_FOLDER, DEFAULT_LOG_FOLDER, DEFAULT_REFERENCE_FOLDER]:
    _d.mkdir(parents=True, exist_ok=True)

# ── Imports ───────────────────────────────────────────────────────────────────────
import io, json, mimetypes, time, wave
import pandas as pd
import requests
import soundfile as sf
from IPython.display import display
from requests.adapters import HTTPAdapter
from scipy import signal
from urllib3.util.retry import Retry

try:
    import noisereduce as nr
except ImportError:
    nr = None

# ── HTTP session factory ──────────────────────────────────────────────────────────
def create_retry_session(max_retries=None):
    n = max_retries or (CONFIG["max_retries"] if "CONFIG" in globals() else 3)
    session = requests.Session()
    retry = Retry(total=n, connect=n, read=n, backoff_factor=2,
                  status_forcelist=[429, 500, 502, 503, 504],
                  allowed_methods=frozenset(["GET", "POST"]), raise_on_status=False)
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session

# ── Configuration ─────────────────────────────────────────────────────────────────
CONFIG = {
    "input_folder":                        DEFAULT_INPUT_FOLDER,
    "output_folder":                       DEFAULT_OUTPUT_FOLDER,
    "cleaned_folder":                      DEFAULT_CLEANED_FOLDER,
    "log_folder":                          DEFAULT_LOG_FOLDER,
    "progress_file":                       DEFAULT_PROGRESS_FILE,
    "reference_folder":                    DEFAULT_REFERENCE_FOLDER,
    "supported_extensions":                {".wav", ".mp3", ".m4a", ".flac"},
    "language":                            "nl",
    "overwrite_outputs":                   False,
    "preprocess_audio":                    True,
    "prefer_cleaned_audio":                True,
    "peak_normalization_target":           0.98,
    "target_sample_rate":                  16000,
    "highpass_hz":                         90,
    "chunk_seconds":                       30,
    "request_timeout":                     300,
    "max_retries":                         3,
    "noise_reduction_strength":            0.8,
    "willma_base_url":                     "https://willma.surf.nl/api/v0",
    "willma_api_key":                      "xxxxxxxxxxxxxxxxxx",
    "diarization_model":                   "pyannote/speaker-diarization-3.1",
    "preferred_whisper_names":             ["whisper-large-v3", "faster-whisper-large-v3", "whisper-large", "whisper-medium"],
    "min_overlap_seconds":                 0.15,
    "alternative_pause_threshold":         1.2,
    "use_api_diarization":                 True,
    "fallback_to_alternative_diarization": True,
    "max_chunk_attempts":                  3,
    "max_diarization_attempts":            2,
    "sleep_between_attempts":              5,
    "stop_after_consecutive_diar_failures":3,
    "max_audio_mb":                        24,
}

print(f"Input:  {DEFAULT_INPUT_FOLDER}\nOutput: {DEFAULT_OUTPUT_FOLDER}")
display(pd.DataFrame([CONFIG]))

Input:  D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHISPER\WAVS
Output: D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHISPER\WAVS\transcriber_v03_outputs


,input_folder,output_folder,cleaned_folder,log_folder,progress_file,reference_folder,supported_extensions,language,overwrite_outputs,preprocess_audio,...,preferred_whisper_names,min_overlap_seconds,alternative_pause_threshold,use_api_diarization,fallback_to_alternative_diarization,max_chunk_attempts,max_diarization_attempts,sleep_between_attempts,stop_after_consecutive_diar_failures,max_audio_mb
0,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,"{.wav, .m4a, .flac, .mp3}",nl,False,True,...,"[whisper-large-v3, faster-whisper-large-v3, wh...",0.15,1.2,True,True,3,2,5,3,24


In [2]:
################ CELL02: OPTIONAL PACKAGE INSTALLATION ################
# Optional install cell
# Uncomment if packages are missing in the active environment.

# %pip install pandas requests soundfile scipy openai-whisper noisereduce

In [3]:
################ CELL02: ALL FUNCTIONS ################

# ── Path and audio discovery ──────────────────────────────────────────────────────
def normalize_path(p):
    return Path(p).expanduser().resolve()

def find_audio_files(folder_path, supported_extensions=None):
    folder = normalize_path(folder_path)
    exts = supported_extensions or CONFIG["supported_extensions"]
    if not folder.exists():
        return []
    return sorted(
        p for p in folder.glob("*")
        if p.is_file() and p.suffix.lower() in exts
    )

def prepare_batch_file_list(folder_path, supported_extensions=None):
    return pd.DataFrame([
        {"file_path": str(p), "file_name": p.name, "extension": p.suffix.lower()}
        for p in find_audio_files(folder_path, supported_extensions)
    ])

# ── Audio validation ──────────────────────────────────────────────────────────────
def validate_audio_file(file_path, supported_extensions=None):
    path = normalize_path(file_path)
    exts = supported_extensions or CONFIG["supported_extensions"]
    r = {
        "file_path": str(path), "exists": path.exists(), "is_file": path.is_file(),
        "supported_extension": path.suffix.lower() in exts,
        "size_bytes": path.stat().st_size if path.exists() and path.is_file() else 0,
        "readable": False, "valid": False, "message": "",
    }
    if not r["exists"]:            r["message"] = "File does not exist."; return r
    if not r["is_file"]:           r["message"] = "Path is not a file."; return r
    if not r["supported_extension"]: r["message"] = "Unsupported audio extension."; return r
    if r["size_bytes"] <= 0:       r["message"] = "File is empty."; return r
    try:
        sf.info(str(path)); r["readable"] = r["valid"] = True; r["message"] = "OK"
    except Exception as exc:
        r["message"] = f"Audio read failed: {exc}"
    return r

def run_preflight_checks(file_paths):
    return pd.DataFrame([validate_audio_file(p) for p in file_paths])

# ── Model catalog ──────────────────────────────────────────────────────────────────
def find_model_by_name_fragment(items, fragments):
    for frag in [(f or "").lower() for f in fragments]:
        for item in items:
            if frag in str(item.get("name", "")).lower():
                return item
    return None

def load_model_catalog(session=None):
    resp = (session or create_retry_session()).get(
        f"{CONFIG['willma_base_url']}/sequences",
        headers={"X-API-KEY": CONFIG["willma_api_key"], "Content-Type": "application/json"},
        timeout=60)
    resp.raise_for_status()
    return resp.json()

def load_whisper_model(session=None, preferred_names=None):
    catalog = load_model_catalog(session=session)
    sel = find_model_by_name_fragment(catalog, preferred_names or CONFIG["preferred_whisper_names"])
    if sel is None:
        sel = next((x for x in catalog if "whisper" in str(x.get("name", "")).lower()), None)
    if sel is None:
        raise ValueError("No Whisper model found in the WILLMA model catalog.")
    CONFIG["selected_whisper_model_name"] = sel.get("name")
    return sel

# ── Transcription ─────────────────────────────────────────────────────────────────
def load_audio_metadata(file_path):
    path = normalize_path(file_path)
    info = sf.info(str(path))
    return {"file_path": str(path), "samplerate": info.samplerate, "channels": info.channels,
            "duration_seconds": round(info.duration, 3), "format": info.format, "subtype": info.subtype}

def overlap_seconds(s_a, e_a, s_b, e_b):
    return max(0.0, min(e_a, e_b) - max(s_a, s_b))

def request_diarization_with_retries(
    session, base_url, api_key, model_name,
    filename, audio_bytes, mime_type, timeout_value, max_attempts, sleep_secs,
):
    last_resp, last_result = None, {}
    for attempt in range(1, max_attempts + 1):
        try:
            last_resp = session.post(
                f"{base_url}/audio/custom-diarization",
                headers={"X-API-KEY": api_key, "Accept": "application/json"},
                data={"model": model_name},
                files={"file": (filename, audio_bytes, mime_type)},
                timeout=timeout_value)
            try:
                last_result = last_resp.json()
            except ValueError:
                last_result = last_resp.text
            if last_resp.status_code < 400 and isinstance(last_result, dict) and not last_result.get("error"):
                return last_resp, last_result
        except Exception as exc:
            last_resp = None
            last_result = {"error": f"Attempt {attempt}: {exc}"}
        if attempt < max_attempts:
            time.sleep(sleep_secs)
    return last_resp, last_result

def _transcribe_bytes(audio_bytes, filename, session, language):
    """POST raw audio bytes to the WILLMA transcription endpoint and return parsed JSON."""
    resp = (session or create_retry_session()).post(
        f"{CONFIG['willma_base_url']}/audio/transcriptions",
        headers={"X-API-KEY": CONFIG["willma_api_key"], "Accept": "application/json"},
        files={"file": (filename, audio_bytes, "audio/wav")},
        data={"model": CONFIG["selected_whisper_model_name"],
              "language": language or CONFIG["language"],
              "stream": "false", "response_format": "verbose_json",
              "timestamp_granularities[]": "segment"},
        timeout=CONFIG["request_timeout"])
    resp.raise_for_status()
    r = resp.json()
    if isinstance(r, dict) and r.get("error"):
        raise RuntimeError(f"WILLMA API error: {r['error']}")
    return r


def _chunk_wav_and_transcribe(path, session, language):
    """Split a large WAV into chunks, transcribe each, and merge with corrected timestamps."""
    import math
    max_bytes = int(CONFIG.get("max_audio_mb", 24) * 1024 * 1024)
    audio, sr  = sf.read(str(path), always_2d=False)
    audio = to_mono(audio)
    bytes_per_sec  = sr * 2                          # 16-bit mono
    chunk_secs     = max(60, max_bytes // bytes_per_sec)
    chunk_samples  = chunk_secs * sr
    n_chunks       = math.ceil(len(audio) / chunk_samples)

    all_segments, text_parts = [], []
    for i in range(n_chunks):
        s0 = i * chunk_samples
        s1 = min(s0 + chunk_samples, len(audio))
        offset = s0 / sr
        buf = io.BytesIO()
        sf.write(buf, audio[s0:s1], sr, subtype="PCM_16", format="WAV")
        log_message(f"{path.name}: chunk {i+1}/{n_chunks} "
                    f"({offset:.1f}s–{s1/sr:.1f}s, {buf.tell()/1024/1024:.1f} MB)")
        r = _transcribe_bytes(buf.getvalue(), path.name, session, language)
        text_parts.append(r.get("text", "").strip())
        for seg in r.get("segments", []):
            adj = dict(seg)
            adj["start"] = round(float(seg.get("start", 0)) + offset, 3)
            adj["end"]   = round(float(seg.get("end",   0)) + offset, 3)
            all_segments.append(adj)

    return {"file_path": str(path), "metadata": load_audio_metadata(path),
            "detected_language": language or CONFIG["language"],
            "text": " ".join(t for t in text_parts if t),
            "segments": all_segments,
            "raw_result": {"chunked": True, "n_chunks": n_chunks}}


def transcribe_single_file(file_path, session=None, language=None):
    path = normalize_path(file_path)
    s    = session or create_retry_session()
    max_bytes = int(CONFIG.get("max_audio_mb", 24) * 1024 * 1024)
    if path.stat().st_size > max_bytes:
        log_message(f"{path.name} is {path.stat().st_size/1024/1024:.1f} MB — splitting into chunks")
        return _chunk_wav_and_transcribe(path, s, language)
    mime = mimetypes.guess_type(path.name)[0] or "application/octet-stream"
    with path.open("rb") as f:
        audio_bytes = f.read()
    r = _transcribe_bytes(audio_bytes, path.name, s, language)
    return {"file_path": str(path), "metadata": load_audio_metadata(path),
            "detected_language": r.get("language", language or CONFIG["language"]),
            "text": r.get("text", "").strip(), "segments": r.get("segments", []), "raw_result": r}

# ── Output formatting ─────────────────────────────────────────────────────────────
def format_srt_timestamp(secs: float) -> str:
    ms = int(round(secs * 1000))
    h, ms = divmod(ms, 3_600_000); m, ms = divmod(ms, 60_000); s, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def prepare_output_paths(file_path, output_folder=None):
    stem = normalize_path(file_path).stem
    out = normalize_path(output_folder or CONFIG["output_folder"])
    out.mkdir(parents=True, exist_ok=True)
    return {"txt":  out / f"{stem}.txt",
            "json": out / f"{stem}.segments.json",
            "csv":  out / f"{stem}.speaker.csv",
            "srt":  out / f"{stem}.speaker.srt"}

def format_segments_as_table(segments):
    return pd.DataFrame(segments) if segments else pd.DataFrame(
        columns=["start", "end", "speaker", "text", "speaker_source", "speaker_overlap"])

def write_txt_transcript(p, text):  Path(p).write_text(str(text or "").strip() + "\n", encoding="utf-8")
def write_json(p, payload):         Path(p).write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
def write_csv(p, segments):         format_segments_as_table(segments).to_csv(p, index=False, encoding="utf-8-sig")

def write_srt(output_path, segments):
    blocks = []; idx = 1
    for seg in segments:
        text = str(seg.get("text", "")).strip()
        if not text:
            continue
        blocks.append(f"{idx}\n{format_srt_timestamp(float(seg.get('start', 0)))} --> "
                      f"{format_srt_timestamp(float(seg.get('end', 0)))}\n"
                      f"[{seg.get('speaker', 'UNKNOWN')}] {text}\n")
        idx += 1
    Path(output_path).write_text("\n".join(blocks), encoding="utf-8")

def save_transcription_outputs(result, output_folder=None):
    paths = prepare_output_paths(result["file_path"], output_folder=output_folder)
    write_txt_transcript(paths["txt"], result.get("text", ""))
    write_json(paths["json"], result)
    write_csv(paths["csv"], result.get("segments", []))
    write_srt(paths["srt"],  result.get("segments", []))
    return paths

# ── Audio preprocessing ───────────────────────────────────────────────────────────
def to_mono(a):
    return a.astype("float32") if getattr(a, "ndim", 1) == 1 else a.mean(axis=1).astype("float32")

def normalize_peak(a, target=0.98):
    peak = float(abs(a).max()) if len(a) else 0.0
    return a if peak <= 0 else ((a / peak) * target).astype("float32")

def preprocess_audio(file_path):
    src = normalize_path(file_path)
    out_dir = normalize_path(CONFIG["cleaned_folder"])
    out_dir.mkdir(parents=True, exist_ok=True)
    dst = out_dir / f"{src.stem}.cleaned.wav"
    if dst.exists() and not CONFIG["overwrite_outputs"]:
        return dst
    audio, sr = sf.read(str(src), always_2d=False)
    audio = to_mono(audio)
    if sr != CONFIG["target_sample_rate"]:
        audio = signal.resample(audio, int(round(len(audio) * CONFIG["target_sample_rate"] / sr))).astype("float32")
        sr = CONFIG["target_sample_rate"]
    b, a = signal.butter(4, min(CONFIG["highpass_hz"] / (sr / 2), 0.99), btype="highpass")
    audio = signal.filtfilt(b, a, audio).astype("float32")
    if nr is not None:
        noise = audio[: min(len(audio), int(sr * 1.5))]
        audio = nr.reduce_noise(y=audio, sr=sr, y_noise=noise,
                                prop_decrease=CONFIG["noise_reduction_strength"],
                                stationary=True).astype("float32")
    sf.write(str(dst), normalize_peak(audio, CONFIG["peak_normalization_target"]), sr, subtype="PCM_16")
    return dst

def choose_transcription_source(original_path, cleaned_path=None):
    orig = normalize_path(original_path)
    cln  = normalize_path(cleaned_path) if cleaned_path else orig.with_name(f"{orig.stem}.cleaned.wav")
    return cln if CONFIG["prefer_cleaned_audio"] and cln.exists() else orig

# ── Diarization ───────────────────────────────────────────────────────────────────
def build_turns_from_stt(segments, pause_threshold=1.2):
    if not segments:
        return []
    segs = sorted(segments, key=lambda x: (float(x.get("start", 0)), float(x.get("end", 0))))
    cur = {"start": float(segs[0].get("start", 0)), "end": float(segs[0].get("end", 0)),
           "text": (segs[0].get("text") or "").strip()}
    turns = []
    for seg in segs[1:]:
        s, e, t = float(seg.get("start", 0)), float(seg.get("end", 0)), (seg.get("text") or "").strip()
        if s - cur["end"] >= pause_threshold:
            turns.append(cur); cur = {"start": s, "end": e, "text": t}
        else:
            cur["end"] = max(cur["end"], e)
            if t:
                cur["text"] = (cur["text"] + " " + t).strip()
    turns.append(cur)
    return turns

def alternative_diarization_from_stt(segments, pause_threshold=1.2, start_with="001"):
    cycle = ["002", "001"] if start_with == "002" else ["001", "002"]
    return [{"start": round(float(t["start"]), 3), "end": round(float(t["end"]), 3), "speaker": cycle[i % 2]}
            for i, t in enumerate(build_turns_from_stt(segments, pause_threshold))]

def force_two_speaker_labels(segments):
    counts = {}
    for seg in segments:
        k = str(seg.get("speaker", "")).strip()
        if k and k != "UNKNOWN":
            counts[k] = counts.get(k, 0) + 1
    top = [x[0] for x in sorted(counts.items(), key=lambda x: x[1], reverse=True)[:2]]
    mapping = {top[i]: f"{i+1:03d}" for i in range(len(top))}
    forced = []
    for seg in segments:
        upd = dict(seg)
        k = str(upd.get("speaker", "")).strip()
        upd["speaker"] = mapping.get(k, k if k in ("001", "002") else "UNKNOWN") if k and k != "UNKNOWN" else "UNKNOWN"
        forced.append(upd)
    return forced

def assign_unknown_by_neighbors(segments):
    s = [dict(x) for x in segments]
    for i, seg in enumerate(s):
        if seg.get("speaker") != "UNKNOWN":
            continue
        prev = next((s[j]["speaker"] for j in range(i-1, -1, -1) if s[j].get("speaker") not in (None, "", "UNKNOWN")), None)
        nxt  = next((s[j]["speaker"] for j in range(i+1, len(s)) if s[j].get("speaker") not in (None, "", "UNKNOWN")), None)
        if prev and prev == nxt:
            seg["speaker"] = prev
    return s

def merge_speaker_segments(segments, max_gap=1.0):
    merged = []
    for seg in sorted(segments, key=lambda x: (float(x.get("start", 0)), float(x.get("end", 0)))):
        text = seg.get("text", "").strip()
        if not text:
            continue
        if not merged:
            merged.append(dict(seg)); continue
        prev = merged[-1]
        if (prev.get("speaker") == seg.get("speaker")
                and float(seg.get("start", 0)) - float(prev.get("end", 0)) <= max_gap):
            prev["end"] = seg.get("end", prev.get("end"))
            prev["text"] = (prev.get("text", "").strip() + " " + text).strip()
        else:
            merged.append(dict(seg))
    return merged

def diarize_audio_or_segments(file_path, transcript_segments, session=None):
    path = normalize_path(file_path)
    mime = mimetypes.guess_type(path.name)[0] or "application/octet-stream"
    _, result = request_diarization_with_retries(
        session or create_retry_session(),
        CONFIG["willma_base_url"], CONFIG["willma_api_key"],
        CONFIG["diarization_model"], path.name, path.read_bytes(), mime,
        CONFIG["request_timeout"], CONFIG["max_diarization_attempts"], CONFIG["sleep_between_attempts"])
    if isinstance(result, dict) and isinstance(result.get("diarization"), list) and result["diarization"]:
        return result["diarization"]
    return alternative_diarization_from_stt(transcript_segments, pause_threshold=CONFIG["alternative_pause_threshold"])

def align_speakers_to_transcript(transcript_segments, diarization_segments):
    aligned = []
    for seg in transcript_segments or []:
        upd = dict(seg)
        s, e = float(upd.get("start", 0)), float(upd.get("end", 0))
        best_sp, best_ov = "UNKNOWN", 0.0
        for d in diarization_segments or []:
            ov = overlap_seconds(s, e, float(d.get("start", 0)), float(d.get("end", 0)))
            if ov > best_ov:
                best_ov = ov
                best_sp = d.get("speaker") or d.get("label") or d.get("id") or "UNKNOWN"
        upd["speaker"] = best_sp if best_ov >= CONFIG["min_overlap_seconds"] else "UNKNOWN"
        upd["speaker_overlap"] = round(best_ov, 3)
        upd["speaker_source"] = "api_or_fallback"
        aligned.append(upd)
    return merge_speaker_segments(assign_unknown_by_neighbors(force_two_speaker_labels(aligned)))

# ── Post-processing and evaluation ────────────────────────────────────────────────
def postprocess_transcript(text):
    c = " ".join(str(text or "").split())
    return (c[0].upper() + c[1:]) if c and c[0].isalpha() else c

def levenshtein_distance(a, b):
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b) + 1))
    for ta in a:
        row = [prev[0] + 1]
        for j, tb in enumerate(b, 1):
            row.append(min(row[-1] + 1, prev[j] + 1, prev[j-1] + (ta != tb)))
        prev = row
    return prev[-1]

def evaluate_transcript(reference_text, predicted_text):
    ref  = " ".join(str(reference_text  or "").strip().lower().split())
    pred = " ".join(str(predicted_text or "").strip().lower().split())
    rw, pw = ref.split(), pred.split()
    return {
        "wer": 0.0 if not rw  and not pw   else levenshtein_distance(rw,  pw)   / max(len(rw),  1),
        "cer": 0.0 if not ref and not pred  else levenshtein_distance(list(ref), list(pred)) / max(len(ref), 1),
    }

# ── Batch processing ──────────────────────────────────────────────────────────────
def log_message(msg, level="INFO"):
    print(f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] [{level}] {msg}")

def summarize_batch_results(results):
    return pd.DataFrame([{
        "file_path": item.get("file_path", ""),
        "status": item.get("status", "unknown"),
        "detected_language": item.get("detected_language", ""),
        "text_length": len(item.get("text", "") or ""),
        "segment_count": len(item.get("segments", []) or []),
        "transcription_audio": item.get("transcription_audio", ""),
        "txt_output": (item.get("output_paths") or {}).get("txt", ""),
        "csv_output": (item.get("output_paths") or {}).get("csv", ""),
        "error": item.get("error", ""),
    } for item in results])

def resume_incomplete_runs(progress_file):
    p = Path(progress_file)
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else []

def save_progress_log(progress_file, results):
    p = Path(progress_file)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")

def process_single_audio_file(file_path, session=None, settings=None):
    settings = settings or CONFIG
    s = session or create_retry_session()
    v = validate_audio_file(file_path, supported_extensions=settings["supported_extensions"])
    if not v["valid"]:
        return {"file_path": str(file_path), "status": "failed", "error": v["message"]}
    src = normalize_path(file_path)
    cleaned = None

    if settings.get("preprocess_audio"):
        try:
            cleaned = preprocess_audio(src)
        except Exception as exc:
            log_message(f"Preprocessing failed for {src.name}: {exc}", level="WARNING")

    tx_src = choose_transcription_source(src, cleaned)
    result = transcribe_single_file(tx_src, session=s, language=settings.get("language"))
    segs = result.get("segments") or []

    if segs:
        merged = align_speakers_to_transcript(segs, diarize_audio_or_segments(tx_src, segs, session=s))
        if merged:
            result["text"] = postprocess_transcript(
                " ".join(x.get("text", "").strip() for x in merged if x.get("text")).strip()
                or result.get("text", ""))
            result["segments"] = merged

    if not result.get("segments"):
        fb = postprocess_transcript(result.get("text", ""))
        result["text"] = fb
        result["segments"] = [{
            "start": 0.0,
            "end": float((result.get("metadata") or {}).get("duration_seconds", 0.0) or 0.0),
            "speaker": "001", "speaker_source": "transcript_fallback",
            "speaker_overlap": 0.0, "text": fb,
        }] if fb else []

    ref = normalize_path(settings["reference_folder"]) / f"{src.stem}.reference.txt"
    result["evaluation"] = (
        evaluate_transcript(ref.read_text(encoding="utf-8"), result["text"]) if ref.exists() else None
    )
    result.update({"status": "ok", "source_audio": str(src),
                   "cleaned_audio": str(cleaned) if cleaned else "",
                   "transcription_audio": str(tx_src)})
    result["output_paths"] = {
        k: str(v) for k, v in save_transcription_outputs(result, output_folder=settings["output_folder"]).items()
    }
    return result

def process_audio_folder(folder_path, session=None, settings=None):
    settings = settings or CONFIG
    s = session or create_retry_session()
    results = []
    for path in find_audio_files(folder_path, supported_extensions=settings["supported_extensions"]):
        log_message(f"Processing {path.name}")
        try:
            results.append(process_single_audio_file(path, session=s, settings=settings))
        except Exception as exc:
            results.append({"file_path": str(path), "status": "failed", "error": str(exc)})
        save_progress_log(settings["progress_file"], results)
    return results


In [4]:
# ── Diagnostic: list all discovered audio files ───────────────────────────────────
found = find_audio_files(CONFIG["input_folder"])
print(f"Found {len(found)} file(s):")
for p in found:
    print(" ", p)


Found 12 file(s):
  D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHISPER\WAVS\1730.wav
  D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHISPER\WAVS\1739.wav
  D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHISPER\WAVS\2344.wav
  D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHISPER\WAVS\3014.wav
  D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHISPER\WAVS\3449.wav
  D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHISPER\WAVS\3629.wav
  D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHISPER\WAVS\4019.wav
  D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHISPER\WAVS\4072.wav
  D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHISPER\WAVS\4284.wav
  D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHISPER\WAVS\4678.wav
  D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHISPER\WAVS\4795.wav
  D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHISPER\WAVS\4925.wav


In [5]:
################ CELL03: RUN — DISCOVER, TRANSCRIBE AND VERIFY ################

# ── Discover audio files and run preflight checks ─────────────────────────────────
audio_file_df = prepare_batch_file_list(CONFIG["input_folder"])
display(audio_file_df.head(20))

preflight_df = (run_preflight_checks(audio_file_df["file_path"].tolist())
                if not audio_file_df.empty else pd.DataFrame())
display(preflight_df.head(20))

# ── Load Whisper model ────────────────────────────────────────────────────────────
session = create_retry_session()
selected_model = load_whisper_model(session=session)
print(f"Selected model: {selected_model.get('name')}")

# ── Transcribe valid files ────────────────────────────────────────────────────────
valid_files = (preflight_df.loc[preflight_df["valid"], "file_path"].tolist()
               if not preflight_df.empty else [])

if not valid_files:
    display(pd.DataFrame([{
        "status": "no_valid_audio_files",
        "input_folder": str(CONFIG["input_folder"]),
        "message": "No valid audio files found.",
    }]))
else:
    print(f"Processing {len(valid_files)} file(s)...")
    batch_results = []
    for fp in valid_files:
        try:
            batch_results.append(process_single_audio_file(fp, session=session, settings=CONFIG))
        except Exception as exc:
            batch_results.append({"file_path": str(fp), "status": "failed", "error": str(exc)})
        save_progress_log(CONFIG["progress_file"], batch_results)
    display(summarize_batch_results(batch_results))

# ── Verify outputs ─────────────────────────────────────────────────────────────────
txt_files = sorted(Path(CONFIG["output_folder"]).glob("*.txt"))[:20]
if txt_files:
    display(pd.DataFrame([{
        "txt":       str(f),
        "txt_exists":  f.exists(),
        "json_exists": f.with_suffix("").with_suffix(".segments.json").exists(),
        "csv_exists":  f.with_suffix("").with_suffix(".speaker.csv").exists(),
        "txt_bytes":   f.stat().st_size if f.exists() else 0,
        "preview":     f.read_text(encoding="utf-8")[:200] if f.exists() else "",
    } for f in txt_files]))

,file_path,file_name,extension
0,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,1730.wav,.wav
1,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,1739.wav,.wav
2,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,2344.wav,.wav
3,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,3014.wav,.wav
4,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,3449.wav,.wav
5,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,3629.wav,.wav
6,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,4019.wav,.wav
7,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,4072.wav,.wav
8,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,4284.wav,.wav
9,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,4678.wav,.wav


,file_path,exists,is_file,supported_extension,size_bytes,readable,valid,message
0,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,True,True,True,160628828,True,True,OK
1,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,True,True,True,141412444,True,True,OK
2,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,True,True,True,185819228,True,True,OK
3,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,True,True,True,159850588,True,True,OK
4,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,True,True,True,197165148,True,True,OK
5,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,True,True,True,317329500,True,True,OK
6,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,True,True,True,85330012,True,True,OK
7,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,True,True,True,78272604,True,True,OK
8,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,True,True,True,32526428,True,True,OK
9,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,True,True,True,49336412,True,True,OK


Selected model: openai/whisper-large-v3
Processing 12 file(s)...
[2026-05-23 00:06:47] [INFO] 1730.cleaned.wav is 51.1 MB — splitting into chunks
[2026-05-23 00:06:47] [INFO] 1730.cleaned.wav: chunk 1/3 (0.0s–786.0s, 24.0 MB)
[2026-05-23 00:07:07] [INFO] 1730.cleaned.wav: chunk 2/3 (786.0s–1572.0s, 24.0 MB)
[2026-05-23 00:07:23] [INFO] 1730.cleaned.wav: chunk 3/3 (1572.0s–1673.2s, 3.1 MB)
[2026-05-23 00:08:27] [INFO] 1739.cleaned.wav is 45.0 MB — splitting into chunks
[2026-05-23 00:08:27] [INFO] 1739.cleaned.wav: chunk 1/2 (0.0s–786.0s, 24.0 MB)
[2026-05-23 00:08:42] [INFO] 1739.cleaned.wav: chunk 2/2 (786.0s–1473.0s, 21.0 MB)
[2026-05-23 00:11:40] [INFO] 2344.cleaned.wav is 59.1 MB — splitting into chunks
[2026-05-23 00:11:40] [INFO] 2344.cleaned.wav: chunk 1/3 (0.0s–786.0s, 24.0 MB)
[2026-05-23 00:11:59] [INFO] 2344.cleaned.wav: chunk 2/3 (786.0s–1572.0s, 24.0 MB)
[2026-05-23 00:12:17] [INFO] 2344.cleaned.wav: chunk 3/3 (1572.0s–1935.6s, 11.1 MB)
[2026-05-23 00:13:00] [INFO] 3014.cl

,file_path,status,detected_language,text_length,segment_count,transcription_audio,txt_output,csv_output,error
0,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,ok,nl,19286,85,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,
1,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,ok,nl,15256,91,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,
2,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,ok,nl,24879,135,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,
3,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,ok,nl,23155,113,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,
4,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,ok,nl,24413,132,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,
5,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,ok,nl,46076,215,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,
6,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,ok,nl,12050,65,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,
7,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,ok,nl,12727,90,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,
8,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,ok,nl,5741,34,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,
9,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,ok,nl,6869,47,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,


,txt,txt_exists,json_exists,csv_exists,txt_bytes,preview
0,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,True,False,False,19320,"Zo, ga lekker zitten. Is dit de eerste keer da..."
1,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,True,False,False,15285,... *Het is de eerste keer dat u hier bent of ...
2,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,True,False,False,24910,"-Het is even kijken, maar voor deze klachten....."
3,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,True,False,False,23193,Eerste keer dat u hier bent? -Ja. Eerste keer....
4,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,True,False,False,24457,"Nog één zat er, ja. Sorry. Ik had geen idee. H..."
5,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,True,False,False,46118,"Dank u wel. Dat zit er op zes, ja. Zullen we e..."
6,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,True,False,False,12076,"Zo, gaat u lekker zitten. De eerste keer dat u..."
7,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,True,False,False,12762,"Ja. En jij bent niet blij? Ik ben blij. Oké, g..."
8,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,True,False,False,5757,Door de enkel gegaan? - Ja. Vertel wat dan. - ...
9,D:\OneDrive - Hogeschool Rotterdam\AA_CODE\WHI...,True,False,False,6882,Ga lekker zitten. Het is niet de eerste keer d...
